In [ ]:
print("Hello, World!")

Finetuning segmentation model with SSL weights...


## Cell 1 — Setup loaders + config

In [2]:
from pathlib import Path
import sys
import torch
from torch.utils.data import DataLoader

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import Kits2DSegDataset
from src.transforms import TransformConfig, SegTransform
from src.train_seg import TrainConfig, train_one_epoch, evaluate, save_checkpoint
from src.models_resnet_unet import ResNet18UNet

DATA_ROOT = Path(r"F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

tfm = SegTransform(TransformConfig(out_size=256, to_3ch=True, normalize_01=True))
train_ds = Kits2DSegDataset(DATA_ROOT, "train", transform=tfm)
val_ds   = Kits2DSegDataset(DATA_ROOT, "val", transform=tfm)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=(DEVICE.type=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=(DEVICE.type=="cuda"))

cfg = TrainConfig(epochs=10, lr=3e-4, dice_weight=0.5, use_amp=True)
cfg

Device: cuda


TrainConfig(lr=0.0003, weight_decay=0.0001, epochs=10, dice_weight=0.5, tumor_class=2, use_amp=True)

## Cell 2 — Train scratch ResNet18-UNet baseline

In [3]:
import torch.optim as optim
import pandas as pd

model = ResNet18UNet(num_classes=3).to(DEVICE)
opt = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))

best = -1
hist = []
ckpt_best = Path("outputs/checkpoints/resnetunet_scratch_best.pt")

for epoch in range(1, cfg.epochs + 1):
    tr = train_one_epoch(model, train_loader, opt, DEVICE, cfg, scaler=scaler)
    va = evaluate(model, val_loader, DEVICE, cfg)
    row = {"epoch": epoch, **tr, **va}
    hist.append(row)
    print(f"[SCRATCH] ep{epoch:02d} train={row['train_loss']:.4f} val={row['val_loss']:.4f} "
          f"tumor_dice={row['tumor_dice']:.4f} tumor_iou={row['tumor_iou']:.4f}")

    if row["tumor_dice"] > best:
        best = row["tumor_dice"]
        save_checkpoint(model, opt, epoch, row, ckpt_best)
        print("  -> saved best", ckpt_best)

df_scratch = pd.DataFrame(hist)
df_scratch

C:\Users\user\AppData\Local\Temp\ipykernel_16036\3601942510.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))


[SCRATCH] ep01 train=0.4815 val=0.3629 tumor_dice=0.3970 tumor_iou=0.3057
  -> saved best outputs\checkpoints\resnetunet_scratch_best.pt
[SCRATCH] ep02 train=0.2038 val=0.3852 tumor_dice=0.3418 tumor_iou=0.2690
[SCRATCH] ep03 train=0.1529 val=0.3575 tumor_dice=0.4119 tumor_iou=0.3275
  -> saved best outputs\checkpoints\resnetunet_scratch_best.pt
[SCRATCH] ep04 train=0.1275 val=0.3578 tumor_dice=0.3862 tumor_iou=0.3076
[SCRATCH] ep05 train=0.1095 val=0.3543 tumor_dice=0.4025 tumor_iou=0.3234
[SCRATCH] ep06 train=0.0968 val=0.3929 tumor_dice=0.3297 tumor_iou=0.2692
[SCRATCH] ep07 train=0.0899 val=0.3619 tumor_dice=0.3803 tumor_iou=0.3060
[SCRATCH] ep08 train=0.0783 val=0.3781 tumor_dice=0.3534 tumor_iou=0.2846
[SCRATCH] ep09 train=0.0771 val=0.3826 tumor_dice=0.3464 tumor_iou=0.2804
[SCRATCH] ep10 train=0.0690 val=0.3568 tumor_dice=0.3924 tumor_iou=0.3204


,epoch,train_loss,val_loss,tumor_dice,tumor_iou,kidney_dice,kidney_iou
0,1,0.481548,0.362924,0.396976,0.305708,0.663230,0.567304
1,2,0.203777,0.385168,0.341750,0.268988,0.700946,0.607526
2,3,0.152942,0.357500,0.411883,0.327528,0.652203,0.557677
3,4,0.127542,0.357776,0.386203,0.307613,0.714600,0.622694
4,5,0.109529,0.354296,0.402487,0.323401,0.709988,0.616707
5,6,0.096832,0.392898,0.329738,0.269160,0.731315,0.641375
6,7,0.089879,0.361911,0.380257,0.305955,0.740403,0.653815
7,8,0.078295,0.378058,0.353369,0.284563,0.730997,0.642293
8,9,0.077106,0.382579,0.346393,0.280423,0.728847,0.641607
9,10,0.068989,0.356831,0.392377,0.320426,0.737004,0.650110


## Cell 3 — Train SSL-initialized ResNet18-UNet

In [4]:
import torch.optim as optim
import pandas as pd

simclr_path = Path("outputs/checkpoints/simclr_resnet18_encoder.pt")
assert simclr_path.exists(), f"Missing: {simclr_path}"

model_ssl = ResNet18UNet(num_classes=3).to(DEVICE)
load_info = model_ssl.load_simclr_encoder(str(simclr_path))
print("Loaded SimCLR encoder:", load_info)

opt_ssl = optim.AdamW(model_ssl.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler_ssl = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))

best = -1
hist = []
ckpt_best = Path("outputs/checkpoints/resnetunet_simclr_best.pt")

for epoch in range(1, cfg.epochs + 1):
    tr = train_one_epoch(model_ssl, train_loader, opt_ssl, DEVICE, cfg, scaler=scaler_ssl)
    va = evaluate(model_ssl, val_loader, DEVICE, cfg)
    row = {"epoch": epoch, **tr, **va}
    hist.append(row)
    print(f"[SIMCLR] ep{epoch:02d} train={row['train_loss']:.4f} val={row['val_loss']:.4f} "
          f"tumor_dice={row['tumor_dice']:.4f} tumor_iou={row['tumor_iou']:.4f}")

    if row["tumor_dice"] > best:
        best = row["tumor_dice"]
        save_checkpoint(model_ssl, opt_ssl, epoch, row, ckpt_best)
        print("  -> saved best", ckpt_best)

df_simclr = pd.DataFrame(hist)
df_simclr

f:\projects\hirdl\FedSSL_Paper\src\models_resnet_unet.py:95: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(simclr_encoder_ckpt, map_location="cpu")
C:\Use

Loaded SimCLR encoder: {'missing': [], 'unexpected': []}
[SIMCLR] ep01 train=0.4455 val=0.3518 tumor_dice=0.4019 tumor_iou=0.3174
  -> saved best outputs\checkpoints\resnetunet_simclr_best.pt
[SIMCLR] ep02 train=0.2003 val=0.3169 tumor_dice=0.4570 tumor_iou=0.3657
  -> saved best outputs\checkpoints\resnetunet_simclr_best.pt
[SIMCLR] ep03 train=0.1508 val=0.3184 tumor_dice=0.4585 tumor_iou=0.3692
  -> saved best outputs\checkpoints\resnetunet_simclr_best.pt
[SIMCLR] ep04 train=0.1292 val=0.2855 tumor_dice=0.5110 tumor_iou=0.4145
  -> saved best outputs\checkpoints\resnetunet_simclr_best.pt
[SIMCLR] ep05 train=0.1099 val=0.3245 tumor_dice=0.4378 tumor_iou=0.3572
[SIMCLR] ep06 train=0.0984 val=0.3652 tumor_dice=0.3810 tumor_iou=0.3089
[SIMCLR] ep07 train=0.0905 val=0.3903 tumor_dice=0.3287 tumor_iou=0.2676
[SIMCLR] ep08 train=0.0837 val=0.3623 tumor_dice=0.3757 tumor_iou=0.3105
[SIMCLR] ep09 train=0.0791 val=0.3892 tumor_dice=0.3403 tumor_iou=0.2760
[SIMCLR] ep10 train=0.0710 val=0.3546 

,epoch,train_loss,val_loss,tumor_dice,tumor_iou,kidney_dice,kidney_iou
0,1,0.445481,0.351782,0.401922,0.317437,0.680530,0.587548
1,2,0.200315,0.316901,0.456960,0.365731,0.697407,0.606526
2,3,0.150833,0.318372,0.458534,0.369195,0.703104,0.609562
3,4,0.129236,0.285538,0.510951,0.414542,0.724328,0.634728
4,5,0.109912,0.324530,0.437776,0.357234,0.726948,0.641552
5,6,0.098393,0.365172,0.381047,0.308858,0.729102,0.643441
6,7,0.090546,0.390288,0.328741,0.267576,0.722433,0.635314
7,8,0.083699,0.362253,0.375680,0.310493,0.722998,0.637324
8,9,0.079099,0.389153,0.340270,0.275955,0.737079,0.650242
9,10,0.071045,0.354627,0.383723,0.319168,0.743181,0.657845


## Create test loader

In [5]:
from torch.utils.data import DataLoader
from pathlib import Path
import torch
import pandas as pd

test_ds = Kits2DSegDataset(DATA_ROOT, "test", transform=tfm)
test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,              # if Windows issues: set 0
    pin_memory=(DEVICE.type=="cuda")
)

print("test size:", len(test_ds))

test size: 3808


## Helper to load checkpoint and evaluate

In [6]:
def eval_ckpt(model_class, ckpt_path: str):
    ckpt_path = Path(ckpt_path)
    assert ckpt_path.exists(), f"Missing: {ckpt_path}"

    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = model_class(num_classes=3).to(DEVICE)
    model.load_state_dict(ckpt["model"])

    val_m = evaluate(model, val_loader, DEVICE, cfg)
    test_m = evaluate(model, test_loader, DEVICE, cfg)

    return ckpt.get("epoch"), ckpt.get("metrics"), val_m, test_m

## Make a comparison table (including your UNet baseline)

In [8]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader

# test loader
test_ds = Kits2DSegDataset(DATA_ROOT, "test", transform=tfm)
test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,  # if Windows issues: 0
    pin_memory=(DEVICE.type=="cuda")
)

def eval_ckpt(model_class, ckpt_path: str):
    ckpt_path = Path(ckpt_path)
    assert ckpt_path.exists(), f"Missing: {ckpt_path}"
    ckpt = torch.load(ckpt_path, map_location=DEVICE)

    model = model_class(num_classes=3).to(DEVICE)
    model.load_state_dict(ckpt["model"])

    val_m = evaluate(model, val_loader, DEVICE, cfg)
    test_m = evaluate(model, test_loader, DEVICE, cfg)
    return ckpt.get("epoch"), ckpt.get("metrics"), val_m, test_m

from src.models_resnet_unet import ResNet18UNet

scratch_ckpt = "outputs/checkpoints/resnetunet_scratch_best.pt"
simclr_ckpt  = "outputs/checkpoints/resnetunet_simclr_best.pt"

ep_s, bestrow_s, val_s, test_s = eval_ckpt(ResNet18UNet, scratch_ckpt)
ep_ssl, bestrow_ssl, val_ssl, test_ssl = eval_ckpt(ResNet18UNet, simclr_ckpt)

print("TEST scratch:", test_s)
print("TEST simclr :", test_ssl)

C:\Users\user\AppData\Local\Temp\ipykernel_16036\3933079778.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=DEVICE)


TEST scratch: {'val_loss': 0.35923061751517926, 'tumor_dice': 0.4415502547841865, 'tumor_iou': 0.36457661939427777, 'kidney_dice': 0.6841244716138857, 'kidney_iou': 0.5932689191150915}
TEST simclr : {'val_loss': 0.32666733874469805, 'tumor_dice': 0.5104673579206985, 'tumor_iou': 0.42352136001645585, 'kidney_dice': 0.7584237233317757, 'kidney_iou': 0.6740169176514251}


In [10]:
import pandas as pd

unet_baseline = {
    "model": "UNet baseline",
    "test_loss": 0.29697420630611854,
    "test_tumor_dice": 0.514398313811937,
    "test_tumor_iou": 0.4473658952693279,
}

resnet_scratch = {
    "model": "ResNetUNet scratch",
    "test_loss": 0.35923061751517926,
    "test_tumor_dice": 0.4415502547841865,
    "test_tumor_iou": 0.36457661939427777,
    "test_kidney_dice": 0.6841244716138857,
    "test_kidney_iou": 0.5932689191150915,
}

resnet_simclr = {
    "model": "ResNetUNet + SimCLR",
    "test_loss": 0.32666733874469805,
    "test_tumor_dice": 0.5104673579206985,
    "test_tumor_iou": 0.42352136001645585,
    "test_kidney_dice": 0.7584237233317757,
    "test_kidney_iou": 0.6740169176514251,
}

df_compare = pd.DataFrame([unet_baseline, resnet_scratch, resnet_simclr])
df_compare

,model,test_loss,test_tumor_dice,test_tumor_iou,test_kidney_dice,test_kidney_iou
0,UNet baseline,0.296974,0.514398,0.447366,NaN,NaN
1,ResNetUNet scratch,0.359231,0.441550,0.364577,0.684124,0.593269
2,ResNetUNet + SimCLR,0.326667,0.510467,0.423521,0.758424,0.674017
